# Hugging Face Sentiment Analysis Web API (FastAPI)

This notebook builds and deploys a **FastAPI** web service using the Hugging Face model [`distilbert/distilbert-base-uncased-finetuned-sst-2-english`](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) for English text sentiment analysis (`POSITIVE` / `NEGATIVE` classification).

## 1. Imports & Configuration

In [ ]:
import logging
from contextlib import asynccontextmanager
from datetime import datetime, timezone
from typing import Any, Dict, Optional

import uvicorn
from fastapi import FastAPI, HTTPException, Request, status
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, field_validator
from transformers import pipeline

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("sentiment-api")

# Hugging Face Model Configuration
MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
TASK_NAME = "sentiment-analysis"
MAX_TEXT_LENGTH = 2000

## 2. Load the Hugging Face Model

In [ ]:
# Global dictionary to store model instance and status
ml_models: Dict[str, Any] = {
    "classifier": None,
    "load_error": None,
}


def load_model() -> None:
    """Loads the Hugging Face sentiment analysis pipeline into memory."""
    if ml_models.get("classifier") is not None:
        return
    try:
        logger.info(f"Loading Hugging Face model '{MODEL_NAME}' for task '{TASK_NAME}'...")
        ml_models["classifier"] = pipeline(
            task=TASK_NAME,
            model=MODEL_NAME,
        )
        ml_models["load_error"] = None
        logger.info("Model loaded successfully.")
    except Exception as exc:
        error_msg = f"Failed to load model '{MODEL_NAME}': {str(exc)}"
        logger.error(error_msg, exc_info=True)
        ml_models["classifier"] = None
        ml_models["load_error"] = error_msg


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Lifespan context manager to ensure the ML model is loaded on startup."""
    load_model()
    yield
    logger.info("Shutting down application...")


# Pre-load the model when executing this cell
load_model()

## 3. Initialize FastAPI App & Input Validation Schemas

In [ ]:
app = FastAPI(
    title="Hugging Face Sentiment Analysis Web API",
    description=(
        "A FastAPI web service that deploys a pre-trained DistilBERT model from "
        "Hugging Face for English text sentiment analysis."
    ),
    version="1.0.0",
    lifespan=lifespan,
)


class PredictionRequest(BaseModel):
    text: str = Field(
        ...,
        description="The input text to analyze for sentiment (1 to 2000 characters).",
        examples=["FastAPI makes deploying Hugging Face models incredibly fast and easy!"],
    )

    @field_validator("text")
    @classmethod
    def validate_text_content(cls, value: str) -> str:
        cleaned = value.strip()
        if not cleaned:
            raise ValueError("Input 'text' cannot be empty or contain only whitespace.")
        if len(cleaned) > MAX_TEXT_LENGTH:
            raise ValueError(
                f"Input 'text' exceeds the maximum allowed length of {MAX_TEXT_LENGTH} characters "
                f"(received {len(cleaned)} characters)."
            )
        return cleaned


class PredictionResult(BaseModel):
    label: str = Field(..., description="Predicted sentiment label (e.g., POSITIVE, NEGATIVE).")
    score: float = Field(..., description="Confidence score between 0.0 and 1.0.")


class PredictionResponse(BaseModel):
    input_text: str = Field(..., description="The validated text that was analyzed.")
    model: str = Field(..., description="The Hugging Face model used for inference.")
    prediction: PredictionResult = Field(..., description="The sentiment prediction output.")
    timestamp: str = Field(..., description="UTC ISO-8601 timestamp of the prediction.")


class HealthResponse(BaseModel):
    status: str = Field(..., description="Operational status of the API ('healthy' or 'unhealthy').")
    model_loaded: bool = Field(..., description="Indicates whether the Hugging Face model is ready.")
    model_name: str = Field(..., description="Name of the configured Hugging Face model.")
    detail: Optional[str] = Field(None, description="Additional status or error details.")
    timestamp: str = Field(..., description="UTC ISO-8601 timestamp of the health check.")

## 4. Custom Error Handling

In [ ]:
@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    """Custom handler for missing inputs, invalid JSON, or failed field validation."""
    errors = []
    for err in exc.errors():
        loc = " -> ".join(str(item) for item in err.get("loc", []))
        msg = err.get("msg", "Invalid input.")
        err_type = err.get("type", "validation_error")
        errors.append({"field": loc, "message": msg, "type": err_type})

    logger.warning(f"Validation error on {request.method} {request.url.path}: {errors}")
    return JSONResponse(
        status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
        content={
            "error": "Validation Error",
            "message": "The request payload is missing required fields or has an invalid format.",
            "details": errors,
        },
    )


@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    """Consistent JSON formatting for raised HTTPExceptions."""
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": "HTTP Exception",
            "message": exc.detail,
            "status_code": exc.status_code,
        },
    )


@app.exception_handler(Exception)
async def unexpected_exception_handler(request: Request, exc: Exception):
    """Fallback handler for any unhandled server errors."""
    logger.error(f"Unhandled exception on {request.method} {request.url.path}: {exc}", exc_info=True)
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={
            "error": "Internal Server Error",
            "message": "An unexpected error occurred while processing your request.",
        },
    )

## 5. API Endpoints (`GET /`, `GET /health`, `POST /predict`)

In [ ]:
@app.get("/", summary="System Introduction", tags=["General"])
async def root() -> Dict[str, Any]:
    """Returns a brief introduction and description of the system's functionality."""
    return {
        "name": "Hugging Face Sentiment Analysis API",
        "version": "1.0.0",
        "description": (
            "This Web API performs sentiment analysis (POSITIVE / NEGATIVE classification) "
            "on English text using a fine-tuned DistilBERT model from Hugging Face."
        ),
        "task": TASK_NAME,
        "model": MODEL_NAME,
        "endpoints": {
            "GET /": "Returns system description and available endpoints.",
            "GET /health": "Checks operational status of the API and ML model.",
            "POST /predict": "Analyzes sentiment of the provided input text.",
            "GET /docs": "Interactive Swagger UI API documentation.",
        },
    }


@app.get(
    "/health",
    response_model=HealthResponse,
    summary="Health Check",
    tags=["Monitoring"],
)
async def health_check():
    """Checks and returns the operational status of the API and the loaded ML model."""
    classifier = ml_models.get("classifier")
    load_error = ml_models.get("load_error")
    now_iso = datetime.now(timezone.utc).isoformat()

    if classifier is None:
        return JSONResponse(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            content={
                "status": "unhealthy",
                "model_loaded": False,
                "model_name": MODEL_NAME,
                "detail": load_error or "Model is not loaded.",
                "timestamp": now_iso,
            },
        )

    return HealthResponse(
        status="healthy",
        model_loaded=True,
        model_name=MODEL_NAME,
        detail="API and Hugging Face model are operational.",
        timestamp=now_iso,
    )


@app.post(
    "/predict",
    response_model=PredictionResponse,
    summary="Predict Sentiment",
    tags=["Inference"],
)
@app.post(
    "/generate",
    response_model=PredictionResponse,
    summary="Predict Sentiment (Alias)",
    tags=["Inference"],
    include_in_schema=False,
)
async def predict_sentiment(payload: PredictionRequest) -> PredictionResponse:
    """
    Receives user text input, validates it, runs inference through the Hugging Face
    sentiment analysis model, and returns the prediction in JSON format.
    """
    classifier = ml_models.get("classifier")
    if classifier is None:
        logger.error("Inference requested, but the ML model is not loaded.")
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="The machine learning model is currently unavailable. Please check /health.",
        )

    try:
        raw_output = classifier(payload.text, truncation=True, max_length=512)
        if not raw_output or not isinstance(raw_output, list):
            raise RuntimeError("Model returned an empty or unexpected output structure.")

        top_result = raw_output[0]
        label = str(top_result.get("label", "UNKNOWN"))
        score = round(float(top_result.get("score", 0.0)), 6)

        return PredictionResponse(
            input_text=payload.text,
            model=MODEL_NAME,
            prediction=PredictionResult(label=label, score=score),
            timestamp=datetime.now(timezone.utc).isoformat(),
        )
    except HTTPException:
        raise
    except Exception as exc:
        logger.error(f"Model inference failed: {exc}", exc_info=True)
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Model inference failed: {str(exc)}",
        ) from exc

## 6. Start the FastAPI Server
Run the cell below to start the Uvicorn server at `http://127.0.0.1:8000` (Swagger UI at `http://127.0.0.1:8000/docs`). While this cell is running, execute `python test_API.py` in a terminal to test the endpoints.

In [ ]:
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()